# Project: Ames Housing Saleprice Prediction
---
Project notebook organisation:<br>
**1 - EDA and Data Cleaning** (current notebook)<br>
[2 - Preprocessing and Feature Engineering](./2_preprocessing_and_feature_engineering.ipynb)<br>
[3 - Model Tuning and Insights](./3_model_tuning_and_insights.ipynb)<br>
[3.1 - Model Performance with Automated Feature Selection](3_(appendix)_automated_feature_selection)<br>
<br>
<br>

In [19]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

sns.set_style('ticks')

pd.set_option('display.max_columns', None)

%matplotlib inline

## Introduction & problem statement
   ---

This project examines a comprehensive housing dataset from the here mentioned ([source](https://www.kaggle.com/c/dsi-us-6-project-2-regression-challenge/overview)). Homeowners looking to increase the value of their homes often spend too much remodelling and don't get the return on investment when selling the house. On the other hand, people looking to buy houses want to get the best house possible given a budget. The goal of this project is to address some of these concerns, specifically:
- what features add the most value to a home, and which hurt home values the most?
- given a set of features, what is the expected sale price of a house? 
- given a budget, what kind of house would one be able to afford?

To answer these questions, we have fitted different linear regression models to the housing data in order to determine the features that are most influential on house price, and those that have the least effect. The performance of the models will be evaluated using the R<sup>2</sup> metric, which is a measure of how much the model is able to explain the variance in the dataset.

Due to the scale of this project, it is split into three Jupyter notebooks: EDA and data cleaning, preprocessing and feature engineering, and model tuning and insights (along with an appendix showing some additional model results where I attempted automated feature selection with the [`featuretools` library](https://github.com/Featuretools/featuretools)). This is the EDA and data cleaning notebook.

### Contents

1. [First impressions](#First-impressions)
2. [Null values](#Null-values)
3. [Feature distributions](#Feature-distributions)
4. [Closer examination of features](#Closer-examinations)
5. [Correlations and collinear features](#Collinear-features)
6. [Possible explanatory variables](#Possible-explanatory-variables)
7. [Conclusion](#Conclusion)

## First Impression
--- 

In [20]:
train = pd.read_csv(r'C:\Users\vsmk3\ds_first_git\train.csv')
test = pd.read_csv(r'C:\Users\vsmk3\ds_first_git\test.csv')
print(f'Train: {train.shape} | Test: {test.shape}')

Train: (1460, 81) | Test: (1459, 80)


In [21]:
train.describe().T

,count,mean,std,min,25%,50%,75%,max
Id,1460.0,730.500000,421.610009,1.0,365.75,730.5,1095.25,1460.0
MSSubClass,1460.0,56.897260,42.300571,20.0,20.00,50.0,70.00,190.0
LotFrontage,1201.0,70.049958,24.284752,21.0,59.00,69.0,80.00,313.0
LotArea,1460.0,10516.828082,9981.264932,1300.0,7553.50,9478.5,11601.50,215245.0
OverallQual,1460.0,6.099315,1.382997,1.0,5.00,6.0,7.00,10.0
OverallCond,1460.0,5.575342,1.112799,1.0,5.00,5.0,6.00,9.0
YearBuilt,1460.0,1971.267808,30.202904,1872.0,1954.00,1973.0,2000.00,2010.0
YearRemodAdd,1460.0,1984.865753,20.645407,1950.0,1967.00,1994.0,2004.00,2010.0
MasVnrArea,1452.0,103.685262,181.066207,0.0,0.00,0.0,166.00,1600.0
BsmtFinSF1,1460.0,443.639726,456.098091,0.0,0.00,383.5,712.25,5644.0


In [45]:
desc = train.describe(include=['O'])

# Append a row containing the list of unique entries for each column
# Calculate the percentage of the most frequent value
desc.loc['top_pct'] = (desc.loc['freq'] / desc.loc['count'] * 100).round(2)
desc.loc['unique_entries'] = train[desc.columns].apply(lambda col: col.dropna().unique().tolist())

desc.T

TypeError: Expected numeric dtype, got object instead.

In [44]:
 train['Exterior2nd'].unique().tolist()

['VinylSd',
 'MetalSd',
 'Wd Shng',
 'HdBoard',
 'Plywood',
 'Wd Sdng',
 'CmentBd',
 'BrkFace',
 'Stucco',
 'AsbShng',
 'Brk Cmn',
 'ImStucc',
 'AsphShn',
 'Stone',
 'Other',
 'CBlock']

In [43]:
 train['Exterior1st'].unique().tolist()

['VinylSd',
 'MetalSd',
 'Wd Sdng',
 'HdBoard',
 'BrkFace',
 'WdShing',
 'CemntBd',
 'Plywood',
 'AsbShng',
 'Stucco',
 'BrkComm',
 'AsphShn',
 'Stone',
 'ImStucc',
 'CBlock']

In [24]:
# for str columns, print count of unique values
# for numerical columns, print % of zeros.

def data_summaries (df):
    for col in df.columns:
        print ("------ %s ------" % col)
        if df[col].dtype != int or float:
            print (df[col].value_counts())
        elif len(df[df[col] == 0]) >= 1026:
            print (col, '\n', '% zeros (HIGH!): ', round((len(df[df[col] == 0])/2051)*100),\
                   '\n', 'no. zeros: ',len(df[df[col]==0]))
        else:
            print (col, '\n', '% zeros: ', round((len(df[df[col] == 0])/2051)*100))
        print ('\n')
    
data_summaries(train)

------ Id ------
Id
1       1
2       1
3       1
4       1
5       1
       ..
1456    1
1457    1
1458    1
1459    1
1460    1
Name: count, Length: 1460, dtype: int64


------ MSSubClass ------
MSSubClass
20     536
60     299
50     144
120     87
30      69
160     63
70      60
80      58
90      52
190     30
85      20
75      16
45      12
180     10
40       4
Name: count, dtype: int64


------ MSZoning ------
MSZoning
RL         1151
RM          218
FV           65
RH           16
C (all)      10
Name: count, dtype: int64


------ LotFrontage ------
LotFrontage
60.0     143
70.0      70
80.0      69
50.0      57
75.0      53
        ... 
182.0      1
160.0      1
152.0      1
153.0      1
46.0       1
Name: count, Length: 110, dtype: int64


------ LotArea ------
LotArea
7200     25
9600     24
6000     17
10800    14
8400     14
         ..
10652     1
12968     1
11924     1
14260     1
9550      1
Name: count, Length: 1073, dtype: int64


------ Street ------
Street
Pave 

## Null values
---

Out of all columns with missing values, electrical and LotFrontage are considered to be true missing values.\
LotFrontage being the horizontal distance to the street connecting to the property, it is assumed that every property has an electrical connection and Lotfrontage associated to it.
- Categorical missing values are replaced with "None" except for in the Electrical column, in which the missing value is replaced by the mode (Anyway only one value is missing in the entire column)
- Numerical missing values are replaced with the median as to not skew the distribution

In [28]:
pd.set_option('display.width', 1000)
pd.set_option('display.max_columns', None)
df=train
missing_summary = pd.DataFrame({
    'Total_Rows': len(df),
    'Non_Null_Count': df.notna().sum(),
    'Missing_Count': df.isna().sum(),
    'Missing_Percent (%)': (df.isna().sum() / len(df) * 100).round(2),
    'dtype': df.dtypes
})

missing_summary = missing_summary[missing_summary['Missing_Count'] > 0].sort_values(
    by='Missing_Count', ascending=False
)

print(missing_summary)

              Total_Rows  Non_Null_Count  Missing_Count  Missing_Percent (%)    dtype
PoolQC              1460               7           1453                99.52   object
MiscFeature         1460              54           1406                96.30   object
Alley               1460              91           1369                93.77   object
Fence               1460             281           1179                80.75   object
MasVnrType          1460             588            872                59.73   object
FireplaceQu         1460             770            690                47.26   object
LotFrontage         1460            1201            259                17.74  float64
GarageType          1460            1379             81                 5.55   object
GarageYrBlt         1460            1379             81                 5.55  float64
GarageFinish        1460            1379             81                 5.55   object
GarageQual          1460            1379             8

In [29]:
cols_with_missing = df.columns[df.isna().any()]

# 2. Separate into categorical and numerical column lists
cat_missing = train[cols_with_missing].select_dtypes(include=['object', 'category']).columns.tolist()
train[cat_missing] = train[cat_missing].fillna("None")
num_missing = train[cols_with_missing].select_dtypes(include=['number']).columns.tolist()
train['LotFrontage'] = train['LotFrontage'].fillna(train['LotFrontage'].median())

train['Electrical'] = train['Electrical'].fillna(train['Electrical'].mode()[0])

## Feature distributions
---

#### Skewed features
The table below shows the top 10 most skewed features and their Fisher-Pearson coefficient of skewness.

In [38]:
from scipy.stats import skew

# list with only numerical columns
col_numerical = [col for col in train.columns if train[col].dtype != object]

# list with only categorical columns
col_categorical = [col for col in train.columns if col not in col_numerical]
train_copy = train.copy()
skewed = train_copy[col_numerical].apply(lambda x: skew(x.astype(float)))
skewed = skewed[skewed > 0.3]
skewed_df = pd.DataFrame({'skewed':skewed})
skewed_df.skewed.sort_values(ascending=False).head(10)

MiscVal          24.451640
PoolArea         14.813135
LotArea          12.195142
3SsnPorch        10.293752
LowQualFinSF      9.002080
KitchenAbvGr      4.483784
BsmtFinSF2        4.250888
ScreenPorch       4.117977
BsmtHalfBath      4.099186
EnclosedPorch     3.086696
Name: skewed, dtype: float64